# Setup
## Dependencies

In [1]:
from pathlib import Path
import sqlite3

## SQLite Setup

In [2]:
master_db_path = Path("../databases/Master_Database.db").resolve()
master_conn = sqlite3.connect(master_db_path)

master_conn.execute("""
   CREATE TABLE IF NOT EXISTS observations (
       object TEXT,
       obs_date TEXT,
       telescope TEXT,
       img_start TEXT NOT NULL,
       img_end TEXT NOT NULL,
       num_exps INTEGER,
       filter TEXT,
       exp_time REAL,
       mosaic_name_f TEXT,
       zpt_mean_f REAL,
       zpt_median_f REAL,
       zpt_mean_weighted_f REAL,
       zpt_stdev_f REAL,
       ztp_err_f REAL,
       mag_f REAL,
       mag_err_f REAL,
       mosaic_name TEXT,
       zpt_mean REAL,
       zpt_median REAL,
       zpt_mean_weighted REAL,
       zpt_stdev REAL,
       ztp_err REAL,
       mag REAL,
       mag_err REAL,
       error_f TEXT,
       error TEXT,
       url_source TEXT NOT NULL,
       group_id INTEGER PRIMARY KEY,
       UNIQUE (url_source, img_start, img_end)     
   )
""")





In [3]:
exposure_db_path = Path("/Users/mehtabsingh/Desktop/Desktop/Durham/3rd Year/Modules/Astro Lab/Astrolab_Code/databases/Durham_Observation_Database.db").resolve()

exposure_conn = sqlite3.connect(exposure_db_path)

def get_overview(exposure_conn):
    sql = """
        SELECT
            CASE
                WHEN UPPER(TRIM(object)) LIKE 'SN%'
                    THEN 'SN' || LOWER(SUBSTR(TRIM(object), 3))
                ELSE 'AT' || LOWER(SUBSTR(TRIM(object), 3))
            END AS target,
            COUNT(DISTINCT obs_date) AS nights,
            COUNT(*) AS exposures,
            MIN(obs_date) AS first_night,
            MAX(obs_date) AS latest_night,
            GROUP_CONCAT(DISTINCT telescope) AS telescopes,
            GROUP_CONCAT(DISTINCT filter) AS filters
        FROM observations
        WHERE UPPER(TRIM(object)) LIKE 'SN%'
            OR UPPER(TRIM(object)) LIKE 'AT%'
        GROUP BY UPPER(TRIM(object))
        HAVING MAX(obs_date) >= ?
        ORDER BY latest_night DESC, target
    """

    return exposure_conn.execute(sql, ("2026-07-01",)).fetchall()

def get_target_details(conn, target_name):
    sql = """
        SELECT
            UPPER(TRIM(object)) AS target,
            TRIM(obs_date) AS date,
            TRIM(telescope) AS telescope,
            TRIM(img_num) AS img_num,
            TRIM(filter) AS filter,
            (exp_time) AS exp_time,
            TRIM(start_time) AS start_time,
            (fwhm) AS fwhm,
            TRIM(url_source) AS url
        FROM observations
        WHERE UPPER(TRIM(object)) = UPPER(?)
        ORDER BY obs_date DESC, telescope, CAST(img_num AS INTEGER) 
    """
    return conn.execute(sql, (target_name,)).fetchall()

In [12]:
def get_image_ranges(exposures):
    numbers = sorted({row[3] for row in exposures}, key=int)

    if not numbers:
        return "-"

    ranges = []
    start = numbers[0]
    previous = numbers[0]

    for number in numbers[1:]:
        if int(number) != int(previous) + 1:
            if start == previous:
                ranges.append(start)
            else:
                ranges.append(f"{start}-{previous}")
 
            start = number
 
        previous = number
 
    # Add the final range.
    if start == previous:
        ranges.append(start)
    else:
        ranges.append(f"{start}-{previous}")

    return ", ".join(ranges)

In [11]:
exposure_db_path = Path("/Users/mehtabsingh/Desktop/Desktop/Durham/3rd Year/Modules/Astro Lab/Astrolab_Code/databases/Durham_Observation_Database.db").resolve()

exposure_conn = sqlite3.connect(exposure_db_path)

sql = """
    SELECT
        CASE
            WHEN UPPER(TRIM(object)) LIKE 'SN%'
                THEN 'SN' || LOWER(SUBSTR(TRIM(object), 3))
            ELSE 'AT' || LOWER(SUBSTR(TRIM(object), 3))
        END AS target
    FROM observations
    WHERE UPPER(TRIM(object)) LIKE 'SN%'
        OR UPPER(TRIM(object)) LIKE 'AT%'
    GROUP BY UPPER(TRIM(object))
    HAVING MAX(obs_date) >= "2026-07-01"
    ORDER BY target
"""

temp_rows = exposure_conn.execute(sql).fetchall()
#print(temp_rows)

list_of_supernovae = []
for row in temp_rows:
    object_name = row[0]
    #print(object_name)
    list_of_supernovae.append(object_name)

print(list_of_supernovae)

a = []

for targets in list_of_supernovae:
    sql = """
        SELECT
            UPPER(TRIM(object)) AS target,
            TRIM(obs_date) AS date,
            TRIM(telescope) AS telescope,
            TRIM(img_num) AS img_num,
            TRIM(filter) AS filter,
            (exp_time) AS exp_time,
            TRIM(start_time) AS start_time,
            (fwhm) AS fwhm,
            TRIM(url_source) AS url
        FROM observations
        WHERE UPPER(TRIM(object)) = UPPER(?)
        ORDER BY obs_date DESC, telescope, CAST(img_num AS INTEGER) 
    """
    a.append(exposure_conn.execute(sql, (targets,)).fetchall())

#print(a)

['AT2026aeiz', 'AT2026aesj', 'AT2026afys', 'SN2026aaiv', 'SN2026acna', 'SN2026acpp', 'SN2026aeiv', 'SN2026aepk', 'SN2026aesj']
